In [7]:
import sys
import pyspark

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)

Python: 3.11.16 (main, Sep  1 2026, 00:14:22) [GCC 12.2.0]
PySpark: 4.0.4


In [8]:
from pyspark.sql import SparkSession  # Importamos SparkSession para crear la aplicación Spark.

spark = SparkSession.builder \
    .appName("PY4_BigData_Docker") \
    .master("local[2]") \
    .config(
        "spark.jars.packages",
        "com.mysql:mysql-connector-j:26.7.0"
    ) \
    .getOrCreate()  # Creamos Spark y cargamos el driver JDBC de MySQL.

print("Spark version:", spark.version)  # Mostramos la versión de Spark.

Spark version: 4.0.4


In [9]:
from getpass import getpass  # Importamos una función para pedir el password sin mostrarlo.

MYSQL_PASSWORD = getpass("MySQL password: ")  # Pedimos el password de MySQL de forma segura.

jdbc_url = "jdbc:mysql://mysql-py4-docker:3306/bigdata_py4"  # Definimos la URL JDBC hacia MySQL dentro de Docker.

jdbc_properties = {
    "user": "root",
    "password": MYSQL_PASSWORD,
    "driver": "com.mysql.cj.jdbc.Driver"
}  # Definimos las propiedades necesarias para la conexión JDBC.

MySQL password: ··········


In [10]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
from pyspark.sql import functions as F

schema = StructType([
    StructField("id_transaccion", IntegerType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("categoria", StringType(), True),
    StructField("monto", DoubleType(), True),
    StructField("canal", StringType(), True),
    StructField("fecha_hora", TimestampType(), True),
])

df_spark = spark.read.csv("data/transacciones_masivas.csv", header=True, schema=schema)

df_limpio = (
    df_spark
    .dropDuplicates(["id_transaccion"])
    .dropna(subset=["monto", "categoria"])
    .filter(F.col("monto") > 0)
    .withColumn("categoria", F.initcap(F.trim(F.col("categoria"))))
)

df_procesado = (
    df_limpio
    .withColumn("total_con_impuestos", F.round(F.col("monto") * 1.15, 2))
    .withColumn("hora", F.hour(F.col("fecha_hora")))
)

resumen_por_categoria = (
    df_procesado
    .groupBy("categoria")
    .agg(
        F.round(F.avg("total_con_impuestos"), 2).alias("promedio_total_con_impuestos"),
        F.count("*").alias("cantidad_transacciones")
    )
    .orderBy(F.desc("promedio_total_con_impuestos"))
)

resumen_por_categoria.show()

+-----------+----------------------------+----------------------+
|  categoria|promedio_total_con_impuestos|cantidad_transacciones|
+-----------+----------------------------+----------------------+
|Electronica|                      173.75|                  9907|
|  Alimentos|                      173.66|                  9787|
| Tecnologia|                      172.88|                  9760|
|   Deportes|                      171.22|                 10024|
|       Ropa|                      171.05|                  9802|
|      Hogar|                      169.71|                  9877|
+-----------+----------------------------+----------------------+



In [11]:
resumen_por_categoria.write.jdbc(
    url=jdbc_url,
    table="resumen_ventas_categoria",
    mode="overwrite",
    properties=jdbc_properties
)

print("Datos escritos correctamente en MySQL.")

Py4JJavaError: An error occurred while calling o99.jdbc.
: com.mysql.cj.jdbc.exceptions.CommunicationsException: Communications link failure

The last packet sent successfully to the server was 0 milliseconds ago. The driver has not received any packets from the server.
	at com.mysql.cj.jdbc.exceptions.SQLError.createCommunicationsException(SQLError.java:165)
	at com.mysql.cj.jdbc.exceptions.SQLExceptionsMapping.translateException(SQLExceptionsMapping.java:55)
	at com.mysql.cj.jdbc.ConnectionImpl.createNewIO(ConnectionImpl.java:840)
	at com.mysql.cj.jdbc.ConnectionImpl.<init>(ConnectionImpl.java:416)
	at com.mysql.cj.jdbc.ConnectionImpl.getInstance(ConnectionImpl.java:237)
	at com.mysql.cj.jdbc.NonRegisteringDriver.connect(NonRegisteringDriver.java:180)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:50)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(Connecti